# 문제 생성 함수 만들기

In [1]:
from glob import glob
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [2]:
def image_quiz(image_path):
    base64_image = encode_image(image_path)
    quiz_prompt = '''
    제공한 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어 주세요.
    정답은 (1)~(4) 중 하나만 해당되도록 출제하세요.
    아래는 예시입니다.
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빵을 사는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입었습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입었습니다.

    정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입었습니다.
    (주의: 정답은 (1)~(4) 중 하나만 선택하도록 출제하세요.)
    ======
    '''

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': quiz_prompt},
                {
                    'type': 'image_url',
                    'image_url': {
                        'url': f'data:image/jpeg;base64,{base64_image}'
                    }
                }
            ]
        }
    ]

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages
    )

    return response.choices[0].message.content

In [3]:
q = image_quiz('./images/busan_dive.jpg')
print(q)

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 전시회에서 사람들이 노트북을 사용하고 있습니다.
- (2) 벽에 여러 개의 핑크색 배너가 보입니다.
- (3) "DIVE 2024 IN BUSAN"이라고 적힌 큰 화면이 있습니다.
- (4) 모든 사람들이 노트북을 사용하고 있습니다.

정답: (4) 일부 사람들이 노트북을 사용하지 않고 있습니다.


In [4]:
txt = ''
no = 1
for g in glob('./images/*.jpg'):
    try:
        q = image_quiz(g)
    except Exception as e:
        print(e)
        continue

    divider = f'## 문제 {no}\n\n'
    print(divider)

    txt += divider
    filename = os.path.basename(g)
    txt += f'![image]({filename})\n\n'

    print(q)
    txt += q + '\n\n--------------------\n\n'

    with open('./images/image_quiz.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    no += 1

## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 많은 사람들이 노트북을 사용하며 앉아 있습니다.
- (2) 행사장은 다양하게 장식되어 있습니다.
- (3) 사람들이 신발을 벗고 자리에서 작업하고 있습니다.
- (4) 천장에 조명이 설치되어 있습니다.

정답: (3) 사람들이 신발을 벗고 자리에서 작업하고 있지 않습니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) "Local Stitch"라는 글자가 보입니다.
- (2) 노란색 조형물이 설치되어 있습니다.
- (3) 조형물은 건물 위에 있습니다.
- (4) 건물에는 여러 개의 창문이 있습니다.

정답: (3) 조형물은 건물 위가 아닌 지면에 설치되어 있습니다.
## 문제 3


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 카페 내부에는 노란색 벽이 보입니다.
- (2) 사람들이 카운터에서 주문을 하고 있습니다.
- (3) 테이블과 의자가 배치된 휴식 공간이 있습니다.
- (4) 창문 밖으로 바다가 보입니다.

정답: (4) 창문 밖으로 바다가 보이는 것이 아니라 나무와 건물이 보입니다.
## 문제 4


죄송합니다. 이미지를 식별할 수 없습니다. 다만 제공된 이미지 설명을 기반으로 퀴즈를 생성할 수 있습니다. 

설명을 바탕으로 퀴즈를 작성하겠습니다.

-----

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 건물은 벽돌로 마감되어 있습니다.
- (2) 1층에는 'PERSONAL COFFEE'라는 간판이 보입니다.
- (3) 건물 옥상에는 테라스가 있습니다.
- (4) 도로에 주황색 원뿔이 보입니다.

정답: (3) 건물 옥상에는 테라스가 아닌 경사진 지붕이 있습니다.
## 문제 5


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 다양한 종류의 빵이 진열되어 있다.
- (2) 가격표에 한글이 적혀 있다.
- (3) 점원이 파란색 유니폼을

# 영어로 문제 출제하기

In [5]:
def image_quiz(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial:
        raise Exception('Failed to generate a quiz.')

    base64_image = encode_image(image_path)

    quiz_prompt = '''
    제공한 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어 주세요.
    정답은 (1)~(4) 중 하나만 해당하도록 출제하세요.
    토익 리스닝 문제 스타일로 문제를 만들어주세요.
    아래는 예시입니다.
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빵을 사는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입었습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입었습니다.

    Listening: Which of the following descriptions of the image is incorrect?
    - (1) It shows people buying bread at a bakery.
    - (2) The person standing at the front is wearing a red shirt.
    - (3) There are people lining up to take a train.
    - (4) The clerk is wearing a yellow T-shirt.

    정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입었습니다.
    (주의: 정답은 (1)~(4) 중 하나만 선택하도록 출제하세요.)
    ======
    '''

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': quiz_prompt},
                {
                    'type': 'image_url',
                    'image_url': {
                        'url': f'data:image/jpeg;base64,{base64_image}'
                    }
                }
            ]
        }
    ]

    try:
        response = client.chat.completions.create(
            model='gpt-4o',
            messages=messages
        )
    except Exception as e:
        print('failed\n' + e)
        return image_quiz(image_path, n_trial+1)

    content = response.choices[0].message.content

    if "Listening:" in content:
        return content, True
    else:
        return image_quiz(image_path, n_trial+1)

In [6]:
txt = ''
no = 1
for g in glob('./images/*.jpg'):
    q, is_succeed = image_quiz(g)

    if not is_succeed:
        continue

    divider = f'## 문제 {no}\n\n'
    print(divider)

    txt += divider
    filename = os.path.basename(g)
    txt += f'![image]({filename})\n\n'

    print(q)
    txt += q + '\n\n-----------------------\n\n'

    with open('./images/image_quiz_english.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    no += 1

## 문제 1


죄송합니다, 사진 속 사람이나 구체적인 내용을 파악할 수 없습니다. 다만, 사진에 근거하여 일반적인 퀴즈를 제공할 수 있습니다.

-----

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 많은 사람들이 한 장소에 모여 있습니다.
- (2) 사람들이 각자 컴퓨터를 사용하고 있습니다.
- (3) 토론회가 열리고 있습니다.
- (4) 이벤트는 실내에서 진행됩니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) Many people are gathered in one place.
- (2) People are using computers individually.
- (3) A discussion forum is taking place.
- (4) The event is held indoors.

정답: (3) 토론회가 열리고 있지는 않습니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 노란색 조형물이 앞에 놓여 있습니다.
- (2) 건물 외벽에는 "Local Stitch"라는 글자가 보입니다.
- (3) 회색 건물이 사진 속에서 가장 큰 부분을 차지합니다.
- (4) 조형물 옆에 벤치가 놓여져 있습니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) A yellow sculpture is placed in the front.
- (2) The building exterior has the words "Local Stitch."
- (3) A gray building occupies the largest part of the photo.
- (4) There is a bench placed next to the sculpture.

정답: (3) 사진에서 가장 큰 부분은 회색 건물이

# TTS로 영어 듣기 평가 문제 만들기

In [7]:
from glob import glob
import json

txt = ''
eng_dict = []
no = 1
for g in glob('./images/*.jpg'):
    q, is_succeed = image_quiz(g)

    if not is_succeed:
        continue

    divider = f'## 문제 {no}\n\n'
    print(divider)

    txt += divider
    filename = os.path.basename(g)
    txt += f'![image]({filename})\n\n'

    print(q)
    txt += q + '\n\n-----------------------\n\n'

    with open('./images/image_quiz_with_eng.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    eng = q.split('Listening: ')[1].split('정답:')[0].strip()

    eng_dict.append({
        'no': no,
        'eng': eng,
        'img': filename
    })

    with open('./images/image_quiz.eng.json', 'w', encoding='utf-8') as f:
        json.dump(eng_dict, f, ensure_ascii=False, indent=4)

    no += 1

## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?

- (1) 많은 사람들이 회의실에서 노트북을 사용하고 있습니다.
- (2) 천장에는 여러 개의 조명이 설치되어 있습니다.
- (3) 벽에는 "DIVE 2024 IN BUSAN"이라는 글자가 보입니다.
- (4) 사람들이 줄을 서서 음식을 받는 모습입니다.

Listening: Which of the following descriptions of the image is incorrect?

- (1) Many people are using laptops in a conference room.
- (2) There are several lights installed on the ceiling.
- (3) The wall displays the text "DIVE 2024 IN BUSAN."
- (4) People are lining up to receive food.

정답: (4) 사람들이 줄을 서서 음식을 받는 모습이 아닙니다.
## 문제 2


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 노란색 조형물이 보입니다.
- (2) 건물에 "Local Stitch"라는 글자가 있습니다.
- (3) 빨간 벽돌 건물이 있습니다.
- (4) 이미지에 있는 모든 건물은 나무로 만들어졌습니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) A yellow sculpture is visible.
- (2) The building has the words "Local Stitch."
- (3) There is a red brick building.
- (4) All the buildings in the image are made of wood.

정답: (4) 이미지에 있는 모든 건물이 나무가 아닌 다양한 재료로 만들어졌습니다.
## 문제 3


Q: 다음 이미지에 대한 

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [9]:
response = client.audio.speech.create(
    model='tts-1-hd',
    voice='alloy',
    input='Hello world! This is a TTS test.'
)

response.write_to_file('hello_world.mp3')

import IPython.display as ipd

ipd.Audio('hello_world.mp3')

In [10]:
voice = 'ash'
mp3_file = 'hello_world_{voice}.mp3'

response = client.audio.speech.create(
    model='tts-1-hd',
    voice=voice,
    input=f"Hello world! I'm {voice}. This is a TTS test."
)

response.write_to_file(mp3_file)

ipd.Audio(mp3_file)

In [11]:
import json

with open('./images/image_quiz_eng.json', 'r', encoding='utf-8') as f:
    eng_dict = json.load(f)

eng_dict

[{'no': 1,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) A lot of people are using laptops.\n- (2) There are many lights installed on the ceiling.\n- (3) The text "DIVE 2024 IN BUSAN" is visible.\n- (4) People are mostly engaged in outdoor activities.',
  'img': 'busan_dive.jpg'},
 {'no': 2,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) There is a yellow sculpture.\n- (2) The words "Local Stitch" can be seen on the building.\n- (3) The sculpture is green.\n- (4) The building walls are made of brick.',
  'img': 'local_stitch.jpg'},
 {'no': 3,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) It shows the interior of a cafe.\n- (2) The wall behind the counter is yellow.\n- (3) There is a clock on the wall.\n- (4) There are chairs and tables.',
  'img': 'local_stitch_terrarosa.jpg'},
 {'no': 4,
  'eng': 'Which of the following descriptions of the image is incorrect?\n- (1) A brick building is

In [16]:
voices = ['alloy', 'ash', 'coral', 'echo', 'fable', 'onyx', 'nova', 'sage', 'shimmer']

for q in eng_dict:
    no = q['no']
    quiz = q['eng']
    quiz = quiz.replace('- (1)', '- One.\t')
    quiz = quiz.replace('- (2)', '- Two.\t')
    quiz = quiz.replace('- (3)', '- Three.\t')
    quiz = quiz.replace('- (4)', '- Four.\t')

    print(no, quiz)

    voice = voices[no % len(voices)]

    response = client.audio.speech.create(
        model = 'tts-1-hd',
        voice=voice,
        input=f'#{no}. {quiz}'
    )

    response.write_to_file(f'./audio/{no}.mp3')

1 Which of the following descriptions of the image is incorrect?
- One.	 A lot of people are using laptops.
- Two.	 There are many lights installed on the ceiling.
- Three.	 The text "DIVE 2024 IN BUSAN" is visible.
- Four.	 People are mostly engaged in outdoor activities.
2 Which of the following descriptions of the image is incorrect?
- One.	 There is a yellow sculpture.
- Two.	 The words "Local Stitch" can be seen on the building.
- Three.	 The sculpture is green.
- Four.	 The building walls are made of brick.
3 Which of the following descriptions of the image is incorrect?
- One.	 It shows the interior of a cafe.
- Two.	 The wall behind the counter is yellow.
- Three.	 There is a clock on the wall.
- Four.	 There are chairs and tables.
4 Which of the following descriptions of the image is incorrect?
- One.	 A brick building is visible.
- Two.	 There is a sign that says PERSONAL COFFEE on the second floor.
- Three.	 There is a cone-shaped orange object in front of the building.
- Fo

In [17]:
ipd.Audio(f'./audio/{no}.mp3')